In [1]:
!pip install gradio

In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.2 MB/s eta 0:00:00


In [7]:
!pip install pypdf


In [9]:
def word_wrap(string, n_chars=72):
    """
    Wraps a long string to a specified number of characters per line.

    Args:
        string (str): The string to wrap.
        n_chars (int, optional): The maximum number of characters per line. Defaults to 72.

    Returns:
        str: The word-wrapped string.
    """
    if len(string) < n_chars:
        return string  # Return the string directly if it's shorter than the character limit
    else:
        # Find the last space before the character limit and insert a newline
        first_part = string[:n_chars].rsplit(' ', 1)[0]
        remaining_string = string[len(first_part) + 1:]
        return first_part + '\n' + \
               word_wrap(remaining_string, n_chars)

from pypdf import PdfReader

reader = PdfReader("/content/repealedfileopen.pdf")
pdf_texts = [p.extract_text().strip() for p in reader.pages]
pdf_texts = [text for text in pdf_texts if text]
print(word_wrap(pdf_texts[0]))

1 
 
THE INDIAN PENAL CODE 
___________ 
ARRANGEMENT OF SECTIONS

__________ 
CHAPTER I 
INTRODUCTION 
PREAMBLE 
SECTIONS 
1. Title and
extent of operation of the Code.  
2. Punishment of offences committed
within India.  
3. Punishment of offences committed beyond, but which
by law may be tried within, India. 
4. Extension of Code to
extra-territorial offences. 
5. Certain laws not to be affected by this
Act. 
CHAPTER II 
GENERAL EXPLANATIONS 
6. Definitions in the Code to
be understood subject to  exceptions.  
7. Sense of expression once
explained.  
8. Gender. 
9. Number. 
10. “Man”. “Woman ”. 
11.
“Person”. 
12. “Public ”. 
13. [Omitted. ]. 
14. “Servant of Government
”. 
15. [Repealed. ]. 
16. [Repealed .] . 
17. “Government ”. 
18.
“India ”. 
19. “Judge ”. 
20. “Court of Justice ”. 
21. “Public servant
”. 
22. “Moveable property ”. 
23. “Wrongful gain”. 
“Wrongful loss”.

Gaining wrongfully/ Losing wrongfully. 
24. “Dishonestly ”. 
25.
“Fraudulently ”. 
26. “Reason to believe ”.

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter  # Import text splitters from langchain

import numpy as np  # Import NumPy for numerical operations

from pypdf import PdfReader  # Import PdfReader to read PDF files

from tqdm import tqdm  # Import tqdm for progress bar visualization
!pip install langchain-huggingface sentence-transformers

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter

character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=20
)

character_split_texts = character_splitter.split_text('\n\n'.join(pdf_texts))
print(word_wrap(character_split_texts[10]))
print(f"\nTotal chunks: {len(character_split_texts)}")

129. Public servant negligently suffering such prisoner to escape.

130. Aiding escape of, rescuing or harbouring such prisoner. 
CHAPTER
VII 
OF OFFENCES RELATING TO THE  ARMY, NAVYAND AIR FORCE 
131.
Abetting mutiny, or attempting to seduce a soldier, sailor or airman
from his duty. 
132. Abetment of mutiny, if mutiny is committed in
consequence th ereof. 
133. Abetment of assault by soldier, sailor or
airman on his sup erior officer, when in execution of his office. 
134.
Abetment of such assault, if the assault is committed. 
135. Abetment
of desertion of soldier, sailor or airman. 
136. Harbouring deserter.

137. Deserter concealed on board merchant vessel through negligence of
master. 
138. Abetment of act of insubordination by soldier, sailor or
airman. 
138A. [Repealed.]. 
139. Persons subject to certain Acts.

140. Wearing garb or carrying token used by soldier, sailor or airman.

CHAPTER VIII 
OF OFFENCES AGAINST THE  PUBLIC TRANQUILLITY 
141.
Unlawful assembly.

Total chunks

In [31]:
!pip install langchain-community

In [32]:
from getpass import getpass
import os

inference_api_key = getpass()  # Enter your Hugging Face API key securely
os.environ["HUGGINGFACEHUB_API_TOKEN"] = inference_api_key

··········


In [35]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Initialize the embedding function to load the model locally
embedding_function = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_texts(character_split_texts, embedding_function)

print(db.index.ntotal)

query = "What does BNS Section 72 talks about ?"

retrieved_documents = db.similarity_search(query)

retrieved_documents

571


[Document(id='a7575fd1-8d19-4bc1-9d66-05d244b34d99', metadata={}, page_content='2. Subs. by Act 40 of 1975, s. 9, for cl. (a) (w.e.f. 6-8-1975).'),
 Document(id='42085a04-6d34-4488-be4d-773089229075', metadata={}, page_content='1. Subs. by Act 10 of 1886, s. 24(1), for section 225A which had been ins. by Act 27 of 1870, s. 9. \n2. Ins. by Act 43 of 1983, s. 2. \n3. Subs. by Act 13 of 2013, s. 4, for “offence under sectio n 376, section 376A, section 376B, section  376C or section 376D” \n(w.e.f. 3-2-2013). \n4. Subs. by Act 22 of 2018, s. 3, for “section 376A, section 376B, section 376C, section 376D” (w.e.f. 21-4-2018).'),
 Document(id='bc672a2b-618c-4c9d-bc07-69eccaa5433f', metadata={}, page_content='5. Cl. (b) omitted by s. 3 and the Sch., ibid. (w.e.f. 1-4-1951). \n6. Subs. by Act 26 of 1955, s. 117 and the Sch., for “transportation for life” (w.e.f. 1-1-1956).'),
 Document(id='d32b2464-49f5-498d-8c9f-b3143777f263', metadata={}, page_content='produced, purchased, kept, imported, ex